# RAG System for Allen Downey's Books

This notebook builds a **Retrieval-Augmented Generation (RAG)** pipeline over the Markdown versions of Allen Downey's free programming books.

The pipeline covers:
1. **Question 2**: Load books, strip empty lines, and chunk with a sliding window
2. **Question 3**: Index all chunks with `minsearch`
3. **Question 4**: Search the index and identify the top result
4. **Question 5**: Build a full RAG pipeline with an LLM and count input tokens
5. **Question 6**: Use structured outputs with a Pydantic response model


In [ ]:
from gitsource import chunk_documents
import os 

In [ ]:
MD_FOLDER = "books_text"

## Question 2: Chunking for RAG

### Step 1: Load and Prepare Documents

Read each `.md` file from `books_text/`, split into lines, and remove blank lines. Each book becomes a dictionary with:
- `filename`: path to the source file (used as the `source` identifier)
- `content`: list of non-empty lines

Reference: [3 ways to remove empty lines from a string in Python](https://www.slingacademy.com/article/python-3-ways-to-remove-empty-lines-from-a-string/)

In [ ]:
target_format = (".md")
books = []
with os.scandir(path=MD_FOLDER) as entries:
    for entry in entries:
        if entry.is_file() and entry.name.endswith(target_format):
            try:
                filename = entry.name
                with open(file=MD_FOLDER + "/" + filename, mode="r") as file:
                    lines = file.readlines()
                    non_empty_lines = [line for line in lines if line.strip()]
                    
                    
                    
            except Exception as e:
                 print(f"{entry.name}: {e}")
                 continue

            dic_book = dict(filename=MD_FOLDER + "/" + filename, content=non_empty_lines)
            books.append(dic_book)
            
    

In [ ]:
len(books)

### Step 2: Chunk with a Sliding Window

Use `chunk_documents` from the `gitsource` package with:
- `size=100`: each chunk contains 100 lines
- `step=50`: the window advances 50 lines at a time (50% overlap between consecutive chunks)

This produces overlapping chunks of ~4,400 characters / ~780 words on average, which is typical for RAG to preserve context across chunk boundaries.

In [ ]:
chunks = chunk_documents(
    documents=books,
    size=100,
    step=50    
)

In [ ]:
len(chunks)

**Question 2**

How many chunks are produced for the "Think Python" book with these settings?

In [ ]:
nr_chunks = 0
lengths = []
for chunk in chunks:
    if chunk.get("filename").endswith("think_python_2e.md"):
        length = len("".join(chunk.get("content")))
        lengths.append(length)
        nr_chunks = nr_chunks + 1

In [ ]:
nr_chunks

In [ ]:
document_chunks = []
for chunk in chunks:
    new_chunk = chunk.copy()
    joined_lines = "".join(new_chunk.get("content", []))
    new_chunk["content"] = joined_lines
    document_chunks.append(new_chunk)

### Step 3: Prepare Documents for Indexing

The search index expects `content` to be a plain string, not a list of lines. Join the lines in each chunk into a single string before fitting the index.

## Question 3 & 4: Indexing and Search with minsearch

Create a `minsearch.Index` over the `content` field (full-text) and the `filename` field (keyword filter). Fit it with all `document_chunks`, then run a test query and inspect which book the top result comes from.

In [ ]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

In [ ]:
index.fit(document_chunks)

In [ ]:
results = index.search("python function definition", num_results=5)

for result in results:
    print(result.get('filename'))

## Question 5: Full RAG Pipeline

Build the complete RAG pipeline:
1. `search(question)`: retrieves the top 5 relevant chunks from the index
2. `build_prompt(question, results)`: formats the question and context into an LLM prompt
3. `llm(prompt, instructions)`: calls `gpt-4o-mini` and returns the answer **and token usage**
4. `rag(query)`: orchestrates all three steps

The `llm` function also returns `response.usage` so we can count input and output tokens for Question 5.

In [ ]:
from openai import OpenAI
import json

In [ ]:
openai_client = OpenAI()

In [ ]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

In [ ]:
def build_prompt(question, search_results):
    print(search_results)
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

In [ ]:
def search(question):
    return index.search(question, num_results=5)

In [ ]:
def llm(
    user_prompt, 
    instructions, 
    model='gpt-4o-mini'):
    
    
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )
   

    return response.output_text, response.usage


In [ ]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer, usage = llm(prompt, instructions)
    return answer, usage


In [ ]:
answer, usage = rag("python function definition")

In [ ]:
usage_unstructured = usage.input_tokens

In [ ]:
usage_unstructured

In [ ]:
print(answer)

## Question 6: Structured Output with Pydantic

Instead of getting a raw text answer, we ask the LLM to return a **structured JSON response** validated against a Pydantic model. This makes the output programmatically usable.

The `RAGResponse` model captures:
- `answer`: the main response in Markdown
- `found_answer`: whether relevant context was found
- `confidence` / `confidence_explanation`: how confident the model is and why
- `answer_type`: category of the answer (`how-to`, `explanation`, etc.)
- `followup_questions`: suggested next questions for the user

The `llm_structured` and `rag_structured` functions mirror the unstructured versions but use `openai_client.responses.parse` with `text_format=output_type`.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

In [ ]:
class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")

In [ ]:
RAGResponse.model_json_schema()

In [ ]:
def llm_structured(
    user_prompt,
    output_type,
    instructions=None,
    model="gpt-4o-mini",
):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return response.output_parsed, response.usage


In [ ]:
def rag_structured(query, output_type=RAGResponse):
    search_results = search(question=query)
    prompt = build_prompt(question=query, search_results=search_results)
    answer, usage = llm_structured(instructions=instructions,
                            user_prompt=prompt,
                           output_type=output_type)
    return answer, usage

### Execute Structured RAG and Inspect Results

Run the structured pipeline with the same query. Then inspect individual fields of the `RAGResponse` object: the answer text, confidence score, explanation, and follow-up questions.

In [ ]:
answer, usage_un = rag_structured("python function definition")

In [ ]:
print(answer.answer)

In [ ]:
answer.confidence

In [ ]:
answer.confidence_explanation

In [ ]:
answer.followup_questions

In [ ]:
usage_structured = usage_un.input_tokens

In [ ]:
usage_structured - usage_unstructured